# EgoLife two-user QA generation demo

This notebook downloads two synchronized EgoLife clips from `lmms-lab/EgoLife`, downloads the matching eye-gaze CSVs from `Wangtwohappy/EgoLife_EyeTracking_EyeGaze`, extracts representative frames in the style of EgoEverything's `frame_extractor.py`, and builds a prompt that forces question generation to require evidence from **two users' videos**.

Reference frame extractor: https://github.com/ElecBeholder/EgoEverything/blob/fix/Generation/frame_extractor.py


## Colab setup: run this first

If you opened this notebook directly from GitHub/Colab, the repo package is not automatically importable. Run the next cell before all other cells.


In [ ]:
# Colab bootstrap. Safe to rerun.
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/XintongYang1010/Long-video-understanding.git'
BRANCH = 'multi-user'
REPO_DIR = Path('/content/Long-video-understanding')

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'pandas', 'opencv-python-headless', 'pillow', 'scikit-learn', 'torch', 'torchvision', 'openai'], check=True)
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=False)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=False)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=False)
    os.chdir(REPO_DIR)
else:
    # Local run: if launched from the notebooks folder, move to repo root.
    cwd = Path.cwd().resolve()
    if cwd.name == 'notebooks' and cwd.parent.name == 'egolife_two_user_qa':
        os.chdir(cwd.parents[1])

repo_root = Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('Working directory:', repo_root)
print('Package exists:', (repo_root / 'egolife_two_user_qa').exists())


## 0. Install notes

Install these if the notebook environment is missing them:

```bash
pip install huggingface_hub pandas opencv-python-headless pillow scikit-learn torch torchvision
```

`torch`, `torchvision`, and `scikit-learn` are only needed for ResNet+KMeans representative-frame clustering. If unavailable, the notebook falls back to evenly spaced timestamps.


In [ ]:
from __future__ import annotations

import json
import math
import os
import shutil
import subprocess
import tempfile
from pathlib import Path
from typing import Any

import cv2
import pandas as pd
from huggingface_hub import hf_hub_download
from PIL import Image

from egolife_two_user_qa.manifest import build_manifest
from egolife_two_user_qa.evidence import summarize_gaze_csv
from egolife_two_user_qa.prompts import build_generation_prompt

VIDEO_DATASET = "lmms-lab/EgoLife"
GAZE_DATASET = "Wangtwohappy/EgoLife_EyeTracking_EyeGaze"
OUT_ROOT = Path("egolife_two_user_qa/outputs/notebook_two_user_demo")
CACHE_DIR = OUT_ROOT / "hf_cache"
FRAME_DIR = OUT_ROOT / "frames"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FRAME_DIR.mkdir(parents=True, exist_ok=True)

print(OUT_ROOT.resolve())

## 1. Build a small aligned manifest

The manifest code looks for clips where both the EgoLife video and the matching EyeGaze CSV exist. For a minimal two-user demo, use two agents on one day and keep only a few clips per agent-day.


In [ ]:
MANIFEST_PATH = OUT_ROOT / "manifest_two_users.json"

manifest = build_manifest(
    output_path=MANIFEST_PATH,
    agents=["A1_JAKE", "A2_ALICE"],
    days=["DAY1"],
    max_per_agent_day=12,
    include_overlays=True,
)

print(manifest["summary"])
manifest["clips"][:2]

## 2. Pick one synchronized two-user timestamp

This selects the first `DAY + time_token` where two users have matching video and gaze files. You can change `WANTED_AGENTS` or directly assign `selected_clips` if you want a specific pair/time.


In [ ]:
from collections import defaultdict

WANTED_AGENTS = {"A1_JAKE", "A2_ALICE"}
groups: dict[tuple[str, str], list[dict[str, Any]]] = defaultdict(list)
for clip in manifest["clips"]:
    if clip["agent_dir"] in WANTED_AGENTS:
        groups[(clip["day"], clip["time_token"])].append(clip)

selected_clips = None
for key, clips in sorted(groups.items()):
    agent_dirs = {c["agent_dir"] for c in clips}
    if WANTED_AGENTS.issubset(agent_dirs):
        selected_clips = sorted([c for c in clips if c["agent_dir"] in WANTED_AGENTS], key=lambda c: c["agent_dir"])
        break

assert selected_clips, "No synchronized two-user clip found. Increase max_per_agent_day or change agents/day."
[(c["agent_dir"], c["day"], c["clip_clock"], c["video_path"], c["gaze_path"]) for c in selected_clips]

## 3. Download the two videos and matching gaze CSVs

This downloads exactly two `.mp4` clips and two EyeGaze `.csv` files. Set `HF_TOKEN` in the environment if Hugging Face asks for authentication.


In [ ]:
def download_hf_file(repo_id: str, repo_path: str, local_subdir: str) -> Path:
    path = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=repo_path,
        local_dir=CACHE_DIR / local_subdir,
        local_dir_use_symlinks=False,
    )
    return Path(path)

for clip in selected_clips:
    clip["local_video_path"] = str(download_hf_file(VIDEO_DATASET, clip["video_path"], "videos"))
    clip["local_gaze_path"] = str(download_hf_file(GAZE_DATASET, clip["gaze_path"], "gaze"))
    if clip.get("overlay_path"):
        clip["local_overlay_path"] = str(download_hf_file(GAZE_DATASET, clip["overlay_path"], "overlays"))

[(c["agent_dir"], c["local_video_path"], c["local_gaze_path"]) for c in selected_clips]

## 4. Summarize eye gaze

Important: this EyeGaze dataset stores Project Aria CPF yaw/pitch/depth values, not image-pixel `gaze_x/gaze_y`. Without Aria calibration/VRS files, we keep gaze as unprojected angular/depth summary and do **not** claim pixel-level fixation.


In [ ]:
for clip in selected_clips:
    clip["gaze_summary"] = summarize_gaze_csv(clip["local_gaze_path"], max_rows=5000)

for clip in selected_clips:
    print("\n", clip["agent_dir"], clip["day"], clip["clip_clock"])
    print(json.dumps({
        "row_count": clip["gaze_summary"].get("row_count"),
        "projection_status": clip["gaze_summary"].get("projection_status"),
        "yaw": clip["gaze_summary"].get("yaw_rads_summary"),
        "pitch": clip["gaze_summary"].get("pitch_rads_summary"),
        "depth": clip["gaze_summary"].get("depth_m_summary"),
    }, indent=2))

## 5. EgoEverything-style representative frame extraction

EgoEverything's `frame_extractor.py` contains two relevant methods:

- `FrameExtractor.extract_frame_at_timestamp`: save a frame at a chosen timestamp.
- `SegmentFeatureExtractor.get_representative_frames`: sample frames, extract ResNet features, cluster with KMeans, and choose frames closest to cluster centers.

The class below mirrors that behavior. If `torch/torchvision/sklearn` are unavailable, it falls back to evenly spaced timestamps.


In [ ]:
class NotebookFrameExtractor:
    def duration(self, video_path: str | Path) -> float:
        cap = cv2.VideoCapture(str(video_path))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        cap.release()
        return total / fps if total > 0 else 30.0

    def extract_frame_at_timestamp(self, video_path: str | Path, timestamp: float, output_path: str | Path) -> Path:
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            raise RuntimeError(f"Cannot open video: {video_path}")
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        frame_num = max(0, min(int(round(timestamp * fps)), max(total - 1, 0)))
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
        ok, frame = cap.read()
        cap.release()
        if not ok:
            raise RuntimeError(f"Cannot read frame at {timestamp}s from {video_path}")
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(output_path), frame)
        return output_path

    def uniform_timestamps(self, video_path: str | Path, n_frames: int = 5) -> list[float]:
        dur = self.duration(video_path)
        return [min(dur - 0.05, dur * (i + 1) / (n_frames + 1)) for i in range(n_frames)]

    def representative_timestamps(self, video_path: str | Path, n_clusters: int = 5, sample_rate: int = 10) -> list[float]:
        try:
            import numpy as np
            import torch
            import torchvision.transforms as transforms
            from sklearn.cluster import KMeans
            from sklearn.preprocessing import StandardScaler
            from torchvision.models import ResNet50_Weights, resnet50
        except Exception as exc:
            print(f"Clustering dependencies unavailable, using uniform frames: {exc}")
            return self.uniform_timestamps(video_path, n_clusters)

        device = "cuda" if torch.cuda.is_available() else "cpu"
        weights = ResNet50_Weights.DEFAULT
        model = resnet50(weights=weights)
        model.fc = torch.nn.Identity()
        model.to(device).eval()
        transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        cap = cv2.VideoCapture(str(video_path))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        features, timestamps = [], []
        frame_idx = 0
        with torch.no_grad():
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                if frame_idx % sample_rate == 0:
                    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    tensor = transform(rgb).unsqueeze(0).to(device)
                    features.append(model(tensor).cpu().numpy()[0])
                    timestamps.append(frame_idx / fps)
                frame_idx += 1
        cap.release()
        if not features:
            return self.uniform_timestamps(video_path, n_clusters)

        features = np.asarray(features)
        k = min(n_clusters, len(features))
        labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(StandardScaler().fit_transform(features))
        chosen = []
        for cluster_id in sorted(set(labels)):
            idxs = np.where(labels == cluster_id)[0]
            center = features[idxs].mean(axis=0)
            closest = idxs[np.argmin(np.linalg.norm(features[idxs] - center, axis=1))]
            chosen.append(float(timestamps[closest]))
        return sorted(chosen)

extractor = NotebookFrameExtractor()

In [ ]:
FRAMES_PER_USER = 5
USE_CLUSTERING = False  # set True to use ResNet+KMeans like EgoEverything; False is faster/reproducible

for clip in selected_clips:
    video_path = Path(clip["local_video_path"])
    timestamps = extractor.representative_timestamps(video_path, FRAMES_PER_USER) if USE_CLUSTERING else extractor.uniform_timestamps(video_path, FRAMES_PER_USER)
    clip_frame_dir = FRAME_DIR / clip["agent_dir"] / clip["day"] / clip["time_token"]
    frames = []
    for i, ts in enumerate(timestamps, 1):
        out_path = clip_frame_dir / f"frame_{i:02d}_{ts:.2f}s.jpg"
        extractor.extract_frame_at_timestamp(video_path, ts, out_path)
        frames.append({"timestamp_seconds": round(ts, 3), "path": str(out_path)})
    clip["frames"] = frames

for clip in selected_clips:
    print(clip["agent_dir"], clip["frames"])


## 6. EgoEverything-style object-centric thought flow

The full EgoEverything `fix` branch is not only frame extraction. Its generation flow is:

1. Load video, summary, and optional gaze tracking.
2. Uniformly sample keyframes from the video.
3. Detect visible objects on sampled keyframes with Gemini.
4. Merge/crop similar objects with CLIP features.
5. Select one key object using gaze-distance weighting.
6. Give the generator a segment list, QA keyframe, selected object, and tools:
   - `REFINE_FRAME(timestamp_second)`
   - `REFINE_SEGMENT(start_second, end_second)`
   - `REQUEST_REVIEW(...)`
7. Require the generator to use extra evidence, cross-verify facts, then submit an MCQ to a reviewer.
8. The reviewer uses `VERIFY_FRAME` / `VERIFY_SEGMENT`, checks ambiguity and wording, and either approves or sends feedback.

The cells below adapt this thought flow to EgoLife **two-user** QA. Because EgoLife EyeGaze CSVs are Aria CPF yaw/pitch/depth rather than pixel `gaze_x/gaze_y`, gaze-object distance is only enabled when projected 2D gaze exists. Otherwise, the notebook keeps the selected object step as manual or VLM-detection based, and it explicitly tells the prompt not to claim pixel fixation.


In [ ]:
import re
from dataclasses import dataclass

@dataclass
class DetectedObject:
    name: str
    gemini_bbox: list[int]  # [ymin, xmin, ymax, xmax], normalized 0-1000
    bbox_xyxy_norm: list[int]  # [x0, y0, x1, y1], normalized 0-1000
    frame_path: str
    timestamp_seconds: float
    user: str

def parse_gemini_detection_text(text: str, frame_path: str, timestamp_seconds: float, user: str) -> list[DetectedObject]:
    """Parse EgoEverything ObjectDetector format: [ymin,xmin,ymax,xmax]:object_name."""
    pattern = r"\[(\d+),\s*(\d+),\s*(\d+),\s*(\d+)\]\s*:?\s*([^\n\r\[]+)"
    objects = []
    for ymin, xmin, ymax, xmax, name in re.findall(pattern, text):
        ymin, xmin, ymax, xmax = map(int, [ymin, xmin, ymax, xmax])
        if not (0 <= ymin < ymax <= 1000 and 0 <= xmin < xmax <= 1000):
            continue
        clean_name = re.sub(r"^[,\s]*|[,}\s]*$", "", name).strip(' \"')
        if not clean_name:
            continue
        objects.append(DetectedObject(
            name=clean_name,
            gemini_bbox=[ymin, xmin, ymax, xmax],
            bbox_xyxy_norm=[xmin, ymin, xmax, ymax],
            frame_path=frame_path,
            timestamp_seconds=timestamp_seconds,
            user=user,
        ))
    return objects

OBJECT_DETECTION_PROMPT = """Please analyze this image and identify all clearly visible objects with their locations.

IMPORTANT REQUIREMENTS:
1. Focus on objects with distinct, recognizable features.
2. Exclude background elements like walls, floors, ceilings, or lighting unless they are specific objects.
3. Only include objects that are clearly visible and well-defined.
4. Provide accurate bounding box coordinates for each object.

For each detected object, provide the results in this exact format:
[ymin,xmin,ymax,xmax]:<object_name>

Coordinates must be normalized integers in range 0-1000.
"""

print(OBJECT_DETECTION_PROMPT[:500])

### 6.1 Object detection slot

EgoEverything calls Gemini for object detection on each sampled keyframe. This notebook keeps that step explicit: paste model detections into `manual_detection_text_by_frame`, or wire this prompt to your own OpenAI-compatible VLM. The parser expects EgoEverything's `[ymin,xmin,ymax,xmax]:object_name` format.


In [ ]:
# Optional: paste Gemini/object-detector outputs here.
# Key = frame path from clip["frames"]; value = lines like [100,200,300,400]:coffee_mug
manual_detection_text_by_frame: dict[str, str] = {}

detected_objects: list[DetectedObject] = []
for clip in selected_clips:
    for frame in clip["frames"]:
        text = manual_detection_text_by_frame.get(frame["path"], "")
        detected_objects.extend(parse_gemini_detection_text(
            text=text,
            frame_path=frame["path"],
            timestamp_seconds=frame["timestamp_seconds"],
            user=clip["agent_name"],
        ))

print(f"Parsed {len(detected_objects)} detected objects")
detected_objects[:3]

### 6.2 Gaze-weighted object sampling, adapted to EgoLife

EgoEverything samples key objects using a Gaussian over 2D distance between gaze point and object center:

`p(object) proportional to exp(-distance^2 / (2 * sigma^2))`

For released EgoLife EyeGaze CSVs, we usually do **not** have calibrated image-pixel gaze. If a future run supplies projected 2D gaze points in `clip['gaze_summary']['projected_gaze_points_sample']`, the helper below can use them. Otherwise it falls back to selecting visible/manual objects and marks the gaze status as unavailable.


In [ ]:
def bbox_center_norm(obj: DetectedObject) -> tuple[float, float]:
    x0, y0, x1, y1 = obj.bbox_xyxy_norm
    return ((x0 + x1) / 2.0, (y0 + y1) / 2.0)

def nearest_projected_gaze_point(clip: dict[str, Any], timestamp_seconds: float) -> tuple[float, float] | None:
    points = (clip.get("gaze_summary") or {}).get("projected_gaze_points_sample") or []
    best = None
    best_dt = float("inf")
    for p in points:
        if "x" not in p or "y" not in p:
            continue
        # If projected points include relative seconds, use them; otherwise keep first usable point.
        t = p.get("relative_timestamp_seconds", p.get("timestamp_seconds"))
        dt = abs(float(t) - timestamp_seconds) if t is not None else 0.0
        if dt < best_dt:
            best_dt = dt
            best = (float(p["x"]), float(p["y"]))
    return best

def egoeverything_gaze_weight(obj: DetectedObject, clip: dict[str, Any], sigma: float = 400.0) -> dict[str, Any]:
    point = nearest_projected_gaze_point(clip, obj.timestamp_seconds)
    if point is None:
        return {"projection_status": "missing_projected_gaze", "distance": None, "weight": 1.0}
    cx, cy = bbox_center_norm(obj)
    # Projected gaze may be pixel coordinates; normalize if width/height are present.
    gx, gy = point
    distance = math.sqrt((cx - gx) ** 2 + (cy - gy) ** 2)
    weight = math.exp(-(distance ** 2) / (2 * sigma ** 2))
    return {"projection_status": "projected", "distance": distance, "weight": weight}

key_objects_by_user: dict[str, dict[str, Any]] = {}
for clip in selected_clips:
    user_objects = [obj for obj in detected_objects if obj.user == clip["agent_name"]]
    scored = []
    for obj in user_objects:
        score = egoeverything_gaze_weight(obj, clip)
        scored.append((score["weight"], obj, score))
    if scored:
        scored.sort(key=lambda x: x[0], reverse=True)
        weight, obj, score = scored[0]
        key_objects_by_user[clip["agent_name"]] = {
            "name": obj.name,
            "gemini_bbox": obj.gemini_bbox,
            "frame_path": obj.frame_path,
            "timestamp_seconds": obj.timestamp_seconds,
            "gaze_sampling": score,
        }
    else:
        # Notebook fallback: still expose a selected-object slot so the prompt follows EgoEverything.
        first_frame = clip["frames"][len(clip["frames"]) // 2]
        key_objects_by_user[clip["agent_name"]] = {
            "name": "manually_selected_or_vlm_detected_object",
            "gemini_bbox": None,
            "frame_path": first_frame["path"],
            "timestamp_seconds": first_frame["timestamp_seconds"],
            "gaze_sampling": {"projection_status": "missing_detection_or_projected_gaze", "distance": None, "weight": 1.0},
        }

json.dumps(key_objects_by_user, indent=2, ensure_ascii=False)

### 6.3 Tool-calling and review prompts, adapted from EgoEverything

The original generator is asked to think through a future everyday scenario, use `REFINE_FRAME` / `REFINE_SEGMENT`, cross-verify facts from at least two independent sources, generate five options, and submit to a reviewer. Below is the same flow adapted to **two EgoLife users**: at least one needed fact must come from each user's stream, and the reviewer must reject single-user-sufficient questions.


In [ ]:
EGOEVERYTHING_TWO_USER_TOOLS = [
    {
        "name": "REFINE_USER_FRAME",
        "description": "Extract or inspect a specific frame from one user's clip at timestamp_second.",
        "parameters": {"user": "Jake/Alice/...", "timestamp_second": "float"},
    },
    {
        "name": "REFINE_USER_SEGMENT",
        "description": "Sample representative frames from one user's clip over a time interval, using ResNet+KMeans representative-frame extraction when available.",
        "parameters": {"user": "Jake/Alice/...", "start_second": "float", "end_second": "float"},
    },
    {
        "name": "REQUEST_TWO_USER_REVIEW",
        "description": "Submit the MCQ, correct answer, evidence timestamps, and per-user evidence for strict review.",
    },
]

def build_egoeverything_two_user_prompt(packet: dict[str, Any]) -> str:
    return f"""## Overall task
Imagine an AR/VR assistant has access to two synchronized EgoLife egocentric clips from different users. Your job is to create one natural daily-life multiple-choice memory question that requires combining both users' videos.

## EgoEverything-inspired thought flow to follow
1. Read the per-user segment/frame evidence, selected key object, and gaze summary.
2. Brainstorm a realistic future scenario where a person asks the assistant to recall something from the shared moment.
3. Use/refine evidence conceptually with these tools: REFINE_USER_FRAME and REFINE_USER_SEGMENT.
4. Cross-verify every key fact with at least two independent evidence sources.
   - For this two-user task, at least one required fact must come from user A and at least one different required fact must come from user B.
   - Do not count duplicate timestamps from the same user as independent two-user evidence.
5. The question or answer must be connected to the selected object(s), as in EgoEverything.
6. Generate five plausible options A-E, exactly one correct.
7. Submit to a strict reviewer. If either single user alone can answer completely, reject or rewrite.

## Hard constraints
- The final question must need BOTH users' videos.
- Do not mention timestamps, video, footage, recording, frames, dataset, or camera in the final question.
- Use gaze only as Aria CPF yaw/pitch/depth unless projection_status is projected; do not claim pixel-level fixation without calibration.
- Avoid unsupported speculation. If evidence is insufficient, return review.status = reject_insufficient_evidence.

## Available conceptual tools
{json.dumps(EGOEVERYTHING_TWO_USER_TOOLS, indent=2, ensure_ascii=False)}

## Evidence packet
{json.dumps(packet, indent=2, ensure_ascii=False)}

## Return JSON only
{{
  "qa_id": "string",
  "scenario": "future everyday use case where the user asks the assistant",
  "question": "natural question requiring both users",
  "options": ["A...", "B...", "C...", "D...", "E..."],
  "correct": "A/B/C/D/E",
  "answer": "exact correct option text",
  "selected_objects_used": {{"UserA": "object/fact", "UserB": "object/fact"}},
  "required_users": ["user1", "user2"],
  "evidence": [
    {{"user": "user1", "needed_fact": "distinct fact from this user's view", "evidence_timestamps": [0.0]}},
    {{"user": "user2", "needed_fact": "distinct fact from this user's view", "evidence_timestamps": [0.0]}}
  ],
  "single_user_answerability": {{"user1": "insufficient because ...", "user2": "insufficient because ..."}},
  "combined_answerability": "sufficient because combining the two views reveals ...",
  "review": {{"status": "draft/pass/reject_single_user_sufficient/reject_insufficient_evidence", "self_check": "..."}}
}}
"""

def build_egoeverything_two_user_review_prompt(qa_item: dict[str, Any], packet: dict[str, Any]) -> str:
    return f"""You are a strict EgoEverything-style QA reviewer for a two-user EgoLife MCQ.

Checklist:
1. Fact verification: key facts are visible/supported, not guessed.
2. Two-user necessity: each required user contributes a distinct necessary fact.
3. Single-user insufficiency: no single user's clip can answer the whole question.
4. Ambiguity: object/person/location descriptions are unambiguous and viewpoint-independent.
5. Wording: no timestamps, video/footage/recording/frame/dataset/camera words in the final question.
6. MCQ: exactly five options, exactly one correct, distractors plausible and similar style.
7. Gaze: no pixel fixation claims unless projection_status is projected.

Evidence packet:
{json.dumps(packet, indent=2, ensure_ascii=False)}

Generated QA:
{json.dumps(qa_item, indent=2, ensure_ascii=False)}

Return JSON only:
{{"review_passed": true, "fact_verification": "PASS/FAIL...", "two_user_check": "PASS/FAIL...", "single_user_check": "PASS/FAIL...", "ambiguity_review": "PASS/FAIL...", "wording": "PASS/FAIL...", "modification_suggestions": ""}}
"""

print(build_egoeverything_two_user_prompt({"demo": "packet goes here"})[:2500])

## 7. Build a two-user evidence packet

This packet is intentionally shaped like the repo's `egolife_two_user_qa.prompts.build_generation_prompt` expects, but now it also carries EgoEverything-style selected objects and tool-flow metadata. The key constraint is still that the generated question must combine one distinct fact from each user and must be impossible to answer completely from either single user alone.


In [ ]:
packet = {
    "evidence_id": "notebook_two_user_demo_001",
    "candidate_type": "egoeverything_style_synchronized_two_user_clips_with_gaze_summary",
    "generation_method": {
        "source": "EgoEverything fix branch Generation pipeline",
        "adapted_steps": [
            "uniform keyframe sampling",
            "optional ResNet+KMeans representative-frame refinement",
            "Gemini-format object detection slot",
            "gaze-weighted key object sampling when projected gaze is available",
            "tool-style REFINE_USER_FRAME / REFINE_USER_SEGMENT evidence refinement",
            "strict reviewer rejecting single-user-sufficient MCQs",
        ],
    },
    "required_users": [clip["agent_name"] for clip in selected_clips],
    "complementarity": {
        "requirement": "The QA must require evidence from both users' videos. A question answerable from only one user's video should be rejected.",
        "strategy": "Use EgoEverything's object-centric QA idea, but require one selected-object/fact from each user's view so the MCQ cannot be answered by either user alone.",
        "gaze_note": "Use gaze only as yaw/pitch/depth summary unless projection_status is projected; do not claim pixel-level fixation without calibration.",
    },
    "clips": [
        {
            "agent_name": clip["agent_name"],
            "agent_dir": clip["agent_dir"],
            "day": clip["day"],
            "clip_clock": clip["clip_clock"],
            "video_url": clip["video_url"],
            "gaze_url": clip["gaze_url"],
            "gaze_summary": clip["gaze_summary"],
            "frames": clip["frames"],
            "selected_key_object": key_objects_by_user.get(clip["agent_name"]),
            "observation": "TODO: Run a VLM over this user's frames and summarize visible people, objects, actions, locations, and gaze/depth cues.",
        }
        for clip in selected_clips
    ],
}

PACKET_PATH = OUT_ROOT / "two_user_evidence_packet.json"
PACKET_PATH.write_text(json.dumps(packet, indent=2, ensure_ascii=False), encoding="utf-8")
print(PACKET_PATH.resolve())

## 8. Generate the strict two-user question-generation prompt

This is the part that enforces your requirement: the prompt demands that the question must need two users' videos, and the JSON schema must include single-user insufficiency and combined-user sufficiency checks.


In [ ]:
baseline_repo_prompt = build_generation_prompt(packet)
generation_prompt = build_egoeverything_two_user_prompt(packet)

# Extra guardrail requested for this notebook.
generation_prompt += """

Additional hard constraint for this notebook:
- The final MCQ must explicitly require comparing or combining evidence from BOTH selected users' videos.
- If either user's video alone is enough, set review.status to "reject_single_user_sufficient".
- Do not write a question about only one wearer, one view, or one clip.
"""

PROMPT_PATH = OUT_ROOT / "two_user_generation_prompt_egoeverything_style.txt"
BASELINE_PROMPT_PATH = OUT_ROOT / "two_user_generation_prompt_repo_baseline.txt"
PROMPT_PATH.write_text(generation_prompt, encoding="utf-8")
BASELINE_PROMPT_PATH.write_text(baseline_repo_prompt, encoding="utf-8")
print(PROMPT_PATH.resolve())
print("Baseline repo prompt:", BASELINE_PROMPT_PATH.resolve())
print(generation_prompt[:3000])

## 9. Run actual agents: observer -> generator -> reviewer

This is the real agent-calling flow. It is off by default because it needs a VLM/LLM endpoint. When `RUN_AGENTS=True`, the notebook runs:

1. **User observer agent for each clip**: looks at that user's extracted frames and writes visual observations.
2. **Question generator agent**: receives both users' observations plus the evidence packet and creates a two-user MCQ.
3. **Reviewer agent**: checks whether the question truly requires both users.
4. If the reviewer fails it, feedback is appended and the generator retries.

You can use OpenAI, OpenRouter, vLLM, SGLang, or any OpenAI-compatible endpoint. For OpenRouter, set `BASE_URL='https://openrouter.ai/api/v1'` and put your key in `OPENAI_API_KEY` or `OPENROUTER_API_KEY`.


In [ ]:
import base64

RUN_AGENTS = False  # set True after configuring API key / endpoint
BASE_URL = os.environ.get('OPENAI_BASE_URL', 'https://openrouter.ai/api/v1')
API_KEY = os.environ.get('OPENAI_API_KEY') or os.environ.get('OPENROUTER_API_KEY') or 'not-needed'
MODEL = os.environ.get('VLM_MODEL', 'google/gemini-2.5-flash')
MAX_AGENT_ROUNDS = 2

def image_to_data_url(path: str | Path) -> str:
    path = Path(path)
    mime = 'image/jpeg' if path.suffix.lower() in {'.jpg', '.jpeg'} else 'image/png'
    b64 = base64.b64encode(path.read_bytes()).decode('utf-8')
    return f'data:{mime};base64,{b64}'

def openai_client():
    from openai import OpenAI
    return OpenAI(base_url=BASE_URL, api_key=API_KEY)

def chat_text(client, messages, *, temperature=0.2, max_tokens=1600) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content or ''

def extract_json_object(text: str) -> dict[str, Any]:
    text = text.strip()
    if text.startswith('```'):
        text = text.strip('`')
        text = text.replace('json\n', '', 1).replace('JSON\n', '', 1)
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1 or end <= start:
        raise ValueError(f'No JSON object found in model output: {text[:500]}')
    return json.loads(text[start:end+1])

print('Endpoint:', BASE_URL)
print('Model:', MODEL)
print('RUN_AGENTS:', RUN_AGENTS)


### 9.1 Observer agents

Each selected EgoLife user gets an observer agent. The observer sees only that user's frames and gaze summary, then returns structured observations. This is what the generator later combines across users.


In [ ]:
def build_observer_messages(clip: dict[str, Any]) -> list[dict[str, Any]]:
    content = []
    for frame in clip['frames']:
        content.append({'type': 'image_url', 'image_url': {'url': image_to_data_url(frame['path'])}})
    text = {
        'task': 'You are a single-user EgoLife observer agent. Summarize only what is visible in this user view.',
        'user': clip['agent_name'],
        'day': clip['day'],
        'clip_clock': clip['clip_clock'],
        'frames': clip['frames'],
        'selected_key_object': key_objects_by_user.get(clip['agent_name']),
        'gaze_summary': clip.get('gaze_summary'),
        'instructions': [
            'Return JSON only.',
            'Describe visible people, objects, actions, locations, spatial relations, and changes across frames.',
            'List facts this user alone can support.',
            'List uncertainties and things this user cannot know.',
            'Do not claim pixel-level gaze unless projection_status is projected.',
        ],
        'schema': {
            'user': clip['agent_name'],
            'visible_summary': 'string',
            'objects': ['object descriptions'],
            'actions': ['action descriptions'],
            'spatial_relations': ['relation descriptions'],
            'supported_facts': ['facts answerable from this user only'],
            'unknowns': ['facts not answerable from this user only'],
            'candidate_question_hooks': ['facts that could combine with another user'],
        },
    }
    content.append({'type': 'text', 'text': json.dumps(text, indent=2, ensure_ascii=False)})
    return [{'role': 'user', 'content': content}]

user_observations = {}
if RUN_AGENTS:
    client = openai_client()
    for clip in selected_clips:
        print('Observer agent:', clip['agent_name'])
        raw = chat_text(client, build_observer_messages(clip), temperature=0.1, max_tokens=1800)
        try:
            obs = extract_json_object(raw)
        except Exception:
            obs = {'user': clip['agent_name'], 'raw_observation': raw}
        user_observations[clip['agent_name']] = obs
        clip['observation'] = obs
else:
    for clip in selected_clips:
        user_observations[clip['agent_name']] = {
            'user': clip['agent_name'],
            'visible_summary': 'DRY RUN: set RUN_AGENTS=True to generate visual observations from this user frames.',
            'supported_facts': [],
            'unknowns': ['No model call was run.'],
            'candidate_question_hooks': [],
        }
        clip['observation'] = user_observations[clip['agent_name']]

print(json.dumps(user_observations, indent=2, ensure_ascii=False)[:4000])


### 9.2 Generator and reviewer agents call each other

The generator sees both user observations and creates a two-user MCQ. The reviewer then checks the output. If the reviewer says it fails, the reviewer feedback is passed back into the generator for another round.


In [ ]:
def build_generator_messages(packet: dict[str, Any], feedback_history: list[dict[str, Any]] | None = None) -> list[dict[str, str]]:
    enriched_packet = dict(packet)
    enriched_packet['observer_agent_outputs'] = user_observations
    prompt = build_egoeverything_two_user_prompt(enriched_packet)
    if feedback_history:
        prompt += '\n\nReviewer feedback from previous failed attempts:\n' + json.dumps(feedback_history, indent=2, ensure_ascii=False)
        prompt += '\nRevise the MCQ so it passes the reviewer. Return JSON only.'
    return [{'role': 'user', 'content': prompt}]

def build_reviewer_messages(qa_item: dict[str, Any], packet: dict[str, Any]) -> list[dict[str, str]]:
    enriched_packet = dict(packet)
    enriched_packet['observer_agent_outputs'] = user_observations
    return [{'role': 'user', 'content': build_egoeverything_two_user_review_prompt(qa_item, enriched_packet)}]

generated_qa = None
review_result = None
feedback_history = []

if RUN_AGENTS:
    client = openai_client()
    for round_idx in range(1, MAX_AGENT_ROUNDS + 1):
        print(f'Generator agent round {round_idx}')
        raw_qa = chat_text(client, build_generator_messages(packet, feedback_history), temperature=0.2, max_tokens=2200)
        generated_qa = extract_json_object(raw_qa)
        print('Generated QA:', json.dumps(generated_qa, indent=2, ensure_ascii=False)[:3000])

        print(f'Reviewer agent round {round_idx}')
        raw_review = chat_text(client, build_reviewer_messages(generated_qa, packet), temperature=0.0, max_tokens=1400)
        review_result = extract_json_object(raw_review)
        print('Review:', json.dumps(review_result, indent=2, ensure_ascii=False))

        if review_result.get('review_passed') is True:
            print('Reviewer passed the QA.')
            break
        feedback_history.append(review_result)
else:
    print('Dry run. Set RUN_AGENTS=True after configuring BASE_URL/API_KEY/MODEL to run observer -> generator -> reviewer.')

if generated_qa is not None:
    (OUT_ROOT / 'generated_two_user_qa.json').write_text(json.dumps(generated_qa, indent=2, ensure_ascii=False), encoding='utf-8')
if review_result is not None:
    (OUT_ROOT / 'review_two_user_qa.json').write_text(json.dumps(review_result, indent=2, ensure_ascii=False), encoding='utf-8')
